# Food Long-Tail OOD Challenge — Vanilla ResNet Baseline

Plainest possible baseline:
1. ResNet-50 ImageNet pretrained, fine-tune on the full training set (no long-tail tricks).
2. Predict the argmax over 101 known classes for every test image.
3. Write `submission.csv`.

Expected behavior: the model can never predict the `unknown` class (101), so the ~25% OOD test samples will all be wrong — capping the achievable Top-1 accuracy at roughly 75%. This baseline establishes the floor; better submissions add long-tail and OOD handling on top.

In [1]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 直接拉取你们的比赛数据
!kaggle competitions download -c ow-food

# 将下载好的压缩包解压到专门的文件夹中
!unzip -q ow-food.zip -d /content/Food-Classification

print("✅ 数据集极速下载与解压完毕！")

100% 853M/853M [00:06<00:00, 137MB/s]

✅ 数据集极速下载与解压完毕！


## 1. Setup

In [2]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

# 数据根目录与提交输出路径
DATA_ROOT = Path('/content/Food-Classification')
OUT_SUB = DATA_ROOT / 'submission(7).csv'

# 已知类数：模型只学 0..100，OOD（101）由后处理判定
NUM_KNOWN = 101
IMG_SIZE = 224          # ResNet 默认输入；图像本身是 256，会被 resize 到 224
BATCH_SIZE = 64
NUM_WORKERS = 4
EPOCHS = 5              # baseline 跑通即可，不追求收敛
LR = 1e-3
SEED = 42

# 自动选设备：优先 CUDA → Apple MPS → CPU
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print('Device:', DEVICE)

torch.manual_seed(SEED); np.random.seed(SEED)

Device: cuda


## 2. Dataset

In [3]:
IMNET_MEAN = [0.485, 0.456, 0.406]
IMNET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMNET_MEAN, IMNET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMNET_MEAN, IMNET_STD),
])

class FoodDataset(Dataset):
    """从 csv (image_id[, label]) + 图像目录加载样本。
    has_label=False 用于测试集，__getitem__ 第二项返回 image_id 以便回填。"""

    def __init__(self, df, img_dir, transform, has_label):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.has_label = has_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.img_dir / f"{row['image_id']}.jpg").convert('RGB')
        img = self.transform(img)
        if self.has_label:
            return img, int(row['label'])
        return img, row['image_id']

# 现在程序知道 DATA_ROOT 是什么了，再安全地读取 CSV
train_df = pd.read_csv(DATA_ROOT / 'train.csv')
test_df = pd.read_csv(DATA_ROOT / 'test.csv')
print(f'✅ 表格读取成功！train: {len(train_df)}, test: {len(test_df)}')

# 加载图片数据集
tr_ds = FoodDataset(train_df, DATA_ROOT / 'train_images' / 'train_images', train_tf, has_label=True)
test_ds = FoodDataset(test_df, DATA_ROOT / 'test_images' / 'test_images', eval_tf, has_label=False)

# pin_memory 仅在 CUDA 下能加速，MPS/CPU 无效
tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
# 推理时 batch 翻倍，无需反向传播显存压力小
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE*2, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
print('✅ 数据流水线组装完毕，可以开始训练了！')

✅ 表格读取成功！train: 32856, test: 20179
✅ 数据流水线组装完毕，可以开始训练了！


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## 3. Model

## 4. Train

In [4]:
import time
import torch
import torch.nn as nn
from torchvision import models
from tqdm.auto import tqdm
import numpy as np

print("🦁 正在加载现代骨干网络 Swin-Tiny (ImageNet-1K V1 权重)...")

# 1. 下载并加载 Swin-Tiny 模型
model = models.swin_t(weights=models.Swin_T_Weights.IMAGENET1K_V1)

# 2. 替换分类头 (Swin 的分类头叫 model.head)
in_features = model.head.in_features
model.head = nn.Linear(in_features, NUM_KNOWN)

# 3. 送入 GPU
model = model.to(DEVICE)
print(f"✅ Swin-Tiny 构建完毕！输出维度: {NUM_KNOWN}")

# =====================================================================
# ⚖️ 温和版长尾平衡权重
# =====================================================================
class_counts = train_df['label'].value_counts().sort_index().values
weights = 1.0 / (np.sqrt(class_counts) + 1.0)
weights = weights / np.sum(weights) * NUM_KNOWN
class_weights = torch.FloatTensor(weights).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
print("✅ 温和平衡交叉熵损失加载完毕！")

# =====================================================================
# 🚀 优化器与学习率调整 (Transformer 适合 2e-4 左右的学习率)
# =====================================================================
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# 开始正式的 5 轮高阶训练
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    total, correct, loss_sum = 0, 0, 0.0
    bar = tqdm(tr_loader, desc=f'Swin Epoch {epoch}/{EPOCHS}', leave=False)

    for x, y in bar:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
        bar.set_postfix(loss=loss_sum/total, acc=correct/total)

    scheduler.step()
    print(f'Epoch {epoch}/{EPOCHS} | Loss {loss_sum/total:.3f} | Train Acc {correct/total:.3f} | {time.time()-t0:.1f}s')


🦁 正在加载现代骨干网络 Swin-Tiny (ImageNet-1K V1 权重)...
Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to /root/.cache/torch/hub/checkpoints/swin_t-704ceda3.pth


100%|██████████| 108M/108M [00:00<00:00, 118MB/s] 


✅ Swin-Tiny 构建完毕！输出维度: 101
✅ 温和平衡交叉熵损失加载完毕！


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Swin Epoch 1/5:   0%|          | 0/514 [00:00<?, ?it/s]

...

Swin Epoch 4/5:   0%|          | 0/514 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c3181cde7a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()Exception ignored in: 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7c3181cde7a0>  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers

    Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
if w.is_alive():    self._shutdown_workers()

    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
        if w.is_alive(): Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7c3181cde7a0>
^
^ ^Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", 

Epoch 4/5 | Loss 0.523 | Train Acc 0.871 | 496.1s


Swin Epoch 5/5:   0%|          | 0/514 [00:00<?, ?it/s]

Epoch 5/5 | Loss 0.348 | Train Acc 0.915 | 495.1s


In [5]:
import torch.nn.functional as F
import numpy as np
from tqdm.auto import tqdm

NUM_CLASSES = NUM_KNOWN  # 101

# ============================================================
# 🎯 核心修改：适配 Swin Transformer 的特征提取函数
# ============================================================
def extract_swin_features(model, x):
    """
    提取 Swin 最后一层 Head 之前的特征向量。
    输入 x: [batch_size, 3, 224, 224]
    输出 feats: [batch_size, 768] (Swin-Tiny 的特征维度是 768)
    """
    # 1. 经过 Swin 的所有前向特征层，输出 shape: [batch_size, 7, 7, 768]
    x = model.features(x)

    # 2. 转换为通道在前，方便做自适应平均池化 -> [batch_size, 768, 7, 7]
    x = x.permute(0, 3, 1, 2)

    # 3. 经过全局平均池化层 -> [batch_size, 768, 1, 1]
    x = model.avgpool(x)

    # 4. 平铺成向量 -> [batch_size, 768]
    x = torch.flatten(x, 1)
    return x


print("正在提取训练集特征，并计算每一类的特征中心...")
model.eval()

# Swin-Tiny 的最终特征维度是 768 维（ResNet 是 2048 维）
features_per_class = [[] for _ in range(NUM_CLASSES)]

with torch.no_grad():
    for x, y in tqdm(tr_loader, desc="Extract train features"):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        feats = extract_swin_features(model, x)  # [batch_size, 768]
        feats = F.normalize(feats, dim=1)         # 特征归一化

        for i in range(x.size(0)):
            label = y[i].item()
            features_per_class[label].append(feats[i].cpu().numpy())

# 计算每一类的中心向量
class_centers = []
for c in range(NUM_CLASSES):
    if len(features_per_class[c]) == 0:
        print(f"警告：类别 {c} 没有训练样本，使用零向量代替")
        center = np.zeros(768, dtype=np.float32)
    else:
        feats_c = np.stack(features_per_class[c], axis=0)
        center = np.median(feats_c, axis=0)
    class_centers.append(center)

class_centers = np.stack(class_centers, axis=0)  # [101, 768]
class_centers = torch.tensor(class_centers, dtype=torch.float32).to(DEVICE)
class_centers = F.normalize(class_centers, dim=1)

print("训练集特征中心计算完成！")
print("class_centers shape:", class_centers.shape)


正在提取训练集特征，并计算每一类的特征中心...


Extract train features:   0%|          | 0/514 [00:00<?, ?it/s]

训练集特征中心计算完成！
class_centers shape: torch.Size([101, 768])


## 5. Predict on test set

In [6]:
import torch
import torch.nn.functional as F
import numpy as np
from tqdm.auto import tqdm
import pandas as pd

OOD_LABEL = 101
OOD_RATIO = 0.24  # 📊 微微调低一点比例（比如24%），给高精度模型释放更多已知类空间

model.eval()
all_ids = []
all_preds = []
all_hybrid_scores = [] # 🌟 存储混合得分

with torch.no_grad():
    for x, ids in tqdm(test_loader, desc="Predict hybrid features"):
        x = x.to(DEVICE, non_blocking=True)

        # 1. 同时拿到 Logits 和 特征向量
        logits = model(x)
        feats = extract_swin_features(model, x)
        feats = F.normalize(feats, dim=1)

        # 2. 计算纯特征相似度
        sims = feats @ class_centers.T  # [batch_size, 101]
        max_sim, pred_sim = sims.max(dim=1)

        # 3. 提取神经网络模型自身的置信度 (Logits 最大值)
        max_logit, pred_logit = logits.max(dim=1)
        # 对 logit 做一个标准正态缩放，使其和相似度处于同一量级
        norm_logit = (max_logit - max_logit.mean()) / (max_logit.std() + 1e-5)

        # 4. 🚀 双剑合璧：混合能量得分机制 (0.7 特征相似度 + 0.3 模型自留置信度)
        hybrid_score = 0.7 * max_sim + 0.3 * torch.sigmoid(norm_logit)

        all_ids.extend(ids)
        all_preds.extend(pred_sim.cpu().numpy().tolist()) # 依然以高质量的特征相似度预测作为类别基准
        all_hybrid_scores.extend(hybrid_score.cpu().numpy().tolist())

# 5. 利用混合得分动态卡出倒数 24% 的真正顽固 OOD
threshold = np.quantile(all_hybrid_scores, OOD_RATIO)

final_preds = []
ood_count = 0
for pred, score in zip(all_preds, all_hybrid_scores):
    if score <= threshold:
        final_preds.append(OOD_LABEL)
        ood_count += 1
    else:
        final_preds.append(pred)

print(f"📊 混合拦截报告：成功改判了 {ood_count} 个低分样本为未知类(101)，占比 {ood_count/len(final_preds)*100:.1f}%")

# 6. 安全导出 CSV
sub = pd.DataFrame({'image_id': all_ids, 'label': final_preds})
order = pd.read_csv(DATA_ROOT / 'test.csv')['image_id'].tolist()
sub = sub.set_index('image_id').loc[order].reset_index()
sub.to_csv(OUT_SUB, index=False)
print(f'✅ 混合动力版 submission.csv 导出成功！')

Predict test features:   0%|          | 0/158 [00:00<?, ?it/s]

## 6. Submission

In [7]:
sub = pd.DataFrame({'image_id': all_ids, 'label': all_preds})
# DataLoader 是 shuffle=False，但 num_workers>0 时返回顺序不保证完全一致
# 这里按 sample_submission.csv 的 image_id 顺序重排，确保提交对齐
order = pd.read_csv(DATA_ROOT / 'test.csv')['image_id'].tolist()
sub = sub.set_index('image_id').loc[order].reset_index()
sub.to_csv(OUT_SUB, index=False)
print(f'Wrote {OUT_SUB} ({len(sub)} rows)')
sub.head()

Wrote /content/Food-Classification/submission(6).csv (20179 rows)


,image_id,label
0,test_00000,89
1,test_00001,63
2,test_00002,49
3,test_00003,101
4,test_00004,52
